### <div style="background-color: #e7f3fe; border-left: 6px solid #2196F3; padding: 15px; border-radius: 4px; color: #0c5460;"> <strong>Imports</strong>
</div>.

In [ ]:
import sys, os
from pathlib import Path

import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import accuracy_score, classification_report, recall_score
from sklearn.utils.class_weight import compute_class_weight
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.config import BATCH_SIZE, DEVICE, SAMPLE_FRAC, EPOCHS, CLASS_NAMES, RANDOM_SEED
from src.dataset import SkinCancerDataset, get_transforms
from src.model import build_model

import time
MAX_TRAIN_MINUTES = 120
start_time = time.time()

In [ ]:
datadir = PROJECT_ROOT / "data/raw"
metadata_path = datadir / "HAM10000_metadata.csv"

df = pd.read_csv(metadata_path)

image_dir_1 = datadir / "HAM10000_images_part_1"
image_dir_2 = datadir / "HAM10000_images_part_2"

image_path_dict = {
    os.path.splitext(f)[0]: str((image_dir_1 / f).relative_to(PROJECT_ROOT))
    for f in os.listdir(image_dir_1)
}

image_path_dict.update({
    os.path.splitext(f)[0]: str((image_dir_2 / f).relative_to(PROJECT_ROOT))
    for f in os.listdir(image_dir_2)
})

df["path"] = df["image_id"].map(image_path_dict)

In [ ]:
torch.cuda.is_available()
torch.cuda.get_device_name(0)

In [ ]:
df.head()

In [ ]:
df.columns

In [ ]:
dataset = SkinCancerDataset(
    df=df,
    transform=get_transforms(),
    sample_frac=SAMPLE_FRAC,
    image_path_col="path"
)

loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

images, labels = next(iter(loader))

print("Image batch shape:", images.shape)
print("Label batch shape:", labels.shape)
print("Labels:", labels[:10])

In [ ]:
print(images.shape)

In [ ]:
# Label mapping
label_map = {
    "akiec": 0,
    "bcc": 1,
    "bkl": 2,
    "df": 3,
    "mel": 4,
    "nv": 5,
    "vasc": 6
}

df["label"] = df["dx"].map(label_map)

# Train / validation split
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=RANDOM_SEED
)

# Datasets
train_dataset = SkinCancerDataset(
    df=train_df,
    transform=get_transforms(train=True),
    sample_frac=SAMPLE_FRAC,
    image_path_col="path"
)

val_dataset = SkinCancerDataset(
    df=val_df,
    transform=get_transforms(train=False),
    sample_frac=1.0,
    image_path_col="path"
)

# Weighted sampler for class imbalance
train_labels = train_dataset.df["label"].values

class_counts = np.bincount(train_labels)
class_sample_weights = 1.0 / class_counts
sample_weights = class_sample_weights[train_labels]
sample_weights = torch.tensor(sample_weights, dtype=torch.float)

weighted_sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

# DataLoaders
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=weighted_sampler
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# Model, loss, optimizer — no weighted loss, sampler handles imbalance
model = build_model().to(DEVICE)
criterion = nn.CrossEntropyLoss()

UNFREEZE_EPOCH = 5
optimizer = optim.Adam(model.parameters(), lr=1e-4)

print(f"Using device: {DEVICE}")

In [ ]:
# Training + validation
num_epochs = EPOCHS
best_preds = None
best_labels = None
best_melanoma_recall = 0.0

# Auto-increment to match final checkpoint versioning
i = 1
while (PROJECT_ROOT / f"outputs/checkpoints/best_melanoma_recall_mobilenetv3_small_v{i}.pt").exists():
    i += 1

best_checkpoint_path = PROJECT_ROOT / f"outputs/checkpoints/best_melanoma_recall_mobilenetv3_small_v{i}.pt"
best_checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

for epoch in range(num_epochs):
    elapsed_minutes = (time.time() - start_time) / 60

    if elapsed_minutes > MAX_TRAIN_MINUTES:
        print("Kill switch triggered. Stopping training.")
        break
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE).long()

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE).long()

            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = accuracy_score(all_labels, all_preds)
    macro_recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
    melanoma_recall = recall_score(
        all_labels,
        all_preds,
        labels=[4],
        average="macro",
        zero_division=0
    )

    print(
        f"Epoch {epoch+1}/{num_epochs} | "
        f"Loss: {avg_loss:.4f} | "
        f"Accuracy: {accuracy:.4f} | "
        f"Macro Recall: {macro_recall:.4f} | "
        f"Melanoma Recall: {melanoma_recall:.4f}"
    )

    if melanoma_recall > best_melanoma_recall:
        best_melanoma_recall = melanoma_recall
        best_preds = all_preds.copy()
        best_labels = all_labels.copy()
        torch.save(model.state_dict(), best_checkpoint_path)
        print(f"Saved new best melanoma recall checkpoint: {best_checkpoint_path}")

In [ ]:
print(f"Using device: {DEVICE}")
model = model.to(DEVICE)

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        outputs = model(images)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = accuracy_score(all_labels, all_preds)
macro_recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)

# melanoma class = 4 based on our label map
melanoma_recall = recall_score(
    all_labels,
    all_preds,
    labels=[4],
    average="macro",
    zero_division=0
)

print(f"Accuracy: {accuracy:.4f}")
print(f"Macro Recall: {macro_recall:.4f}")
print(f"Melanoma Recall: {melanoma_recall:.4f}")

In [ ]:
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

In [ ]:
report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES)
print(report)

# Auto-increment report filename to match checkpoint versioning
i = 1
while (PROJECT_ROOT / f"outputs/reports/classification_report_v{i}.txt").exists():
    i += 1

report_path = PROJECT_ROOT / f"outputs/reports/classification_report_v{i}.txt"
report_path.parent.mkdir(parents=True, exist_ok=True)

with open(report_path, "w") as f:
    f.write(f"EPOCHS: {EPOCHS} | SAMPLE_FRAC: {SAMPLE_FRAC}\n")
    f.write(f"Total runtime: {elapsed/60:.1f} minutes\n\n")
    f.write(report)

print(f"Saved report to: {report_path}")

In [ ]:
best_report = classification_report(best_labels, best_preds, target_names=CLASS_NAMES)
print("Best Model Classification Report")
print(best_report)

# Match versioning to checkpoint
i = 1
while (PROJECT_ROOT / f"outputs/reports/best_classification_report_v{i}.txt").exists():
    i += 1

best_report_path = PROJECT_ROOT / f"outputs/reports/best_classification_report_v{i}.txt"
best_report_path.parent.mkdir(parents=True, exist_ok=True)

with open(best_report_path, "w") as f:
    f.write(f"EPOCHS: {EPOCHS} | SAMPLE_FRAC: {SAMPLE_FRAC}\n")    
    f.write("Best Model Classification Report\n\n")
    f.write(best_report)

print(f"Saved report to: {best_report_path}")

In [ ]:
checkpoint_dir = PROJECT_ROOT / "outputs/checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

# Auto-increment filename
i = 1
while (checkpoint_dir / f"local_poc_mobilenetv3_small_v{i}.pt").exists():
    i += 1

checkpoint_path = checkpoint_dir / f"local_poc_mobilenetv3_small_v{i}.pt"

torch.save(model.state_dict(), checkpoint_path)
print(f"Saved checkpoint to: {checkpoint_path}")



In [ ]:
# Reload best checkpoint and verify
print(f"Loading from: {best_checkpoint_path}")
model.load_state_dict(torch.load(best_checkpoint_path, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(DEVICE)
        labels = labels.to(DEVICE).long()
        preds = torch.argmax(model(images), dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

reloaded_melanoma_recall = recall_score(all_labels, all_preds, labels=[4], average="macro", zero_division=0)

print(f"Expected (best training recall): {best_melanoma_recall:.4f}")
print(f"Reloaded checkpoint:             {reloaded_melanoma_recall:.4f}")
print(f"Match: {abs(reloaded_melanoma_recall - best_melanoma_recall) < 0.0001}")

In [ ]:
elapsed = time.time() - start_time
print(f"Total runtime: {elapsed/60:.1f} minutes")